In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),  # Resize all images to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # value for each channel ## last step after to tensor

])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
## this is how i can covert letters to start from 1 to 26  but I will not do it bellow because I want the letters
## to start from 0-25 so it matches the creoss entropy loss which start from 0-25
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
for l in letters:
  letter_to_idx = {cls_name: l+1 for l, cls_name in enumerate(letters)}
letter_to_idx

In [ ]:
# Letter mapping (labels are 1-26 for A-Z) ce 0-25 1-26
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
## loop all leters for each map it to number 1-26 and
for l in letters:
  letter_to_idx = {cls_name: l for l, cls_name in enumerate(letters)} ## l+1 to start from 1 until 26 instead from 0 -25 ## I remove+1 so i match the creoss entropy loss
letter_to_idx


# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
letter_to_idx


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
import torchvision.models as models

# Write your code here
# Load pretrained EfficientNetV2-S model
# device = "cuda" if torch.cuda.is_available() else "cpu"
model = efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
# model.eval().to(device) ## eval mode cuz no train this is grouping of the image an unsupervised problem

num_classes = 26  # Pascal VOC has 20 object classes
in_features = model.classifier[1].in_features  # Input features for predictor
model.classifier[1] = nn.Linear(in_features, num_classes)

# Freeze the backbone (feature extractor) - we only want to train the classifier head
model.requires_grad_(False) ## freeze the weights
model.classifier = model.classifier.requires_grad_(True)


# (classifier): Sequential(
  #   (0): Dropout(p=0.2, inplace=True)
  #   (1): Linear(in_features=1280, out_features=1000, bias=True)
  # )
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model  ## >     (1): Linear(in_features=1280, out_features=26, bias=True) now 26 output

In [ ]:
# Hint: EMNIST letters labels are 1-26, but crossentropy starts from 0! I make my letters start from 0-25 to match it
from tqdm import tqdm    # Shows progress bar
## Define the training loop function
# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels-1)  # Compute loss

        ## the learning step
        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

## Define the validation loop function

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs,  labels-1)  # Compute loss ## to match my outputs
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class ## the one with highest probability ## we have more than two classes
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.AdamW(model.parameters(), lr=0.001)  # AdamW optimizer
num_epochs = 5 # Number of epochs ## dont make it too high cuz u dont want long time to get output


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
import numpy as np
@torch.no_grad()
def show_predictions(model, dataset, n=12):
    model.eval()
    fig, axes = plt.subplots(2, n//2, figsize=(15, 4))
    axes = axes.ravel()

    for i in range(n):
        sample = dataset[np.random.randint(0, len(dataset))]
        x, y = sample['image'], sample['label']
        logits = model(x.unsqueeze(0).to(device))
        pred = torch.argmax(logits, dim=1).item()

        x_vis = (x * 0.5) + 0.5
        axes[i].imshow(x_vis.squeeze(0), cmap="gray")
        axes[i].set_title(f"true = {letter_to_idx[y]}, pred = {letter_to_idx[pred]}", fontsize=9)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset, n=12)


In [ ]:
# Write your code here
